# E12 — Conformalized Quantile Regression (CQR)

**Experiment ID:** `E12`. **Spec:** `EXPERIMENT_PLAN.md` §E12. **Governing rules:** `CLAUDE.md`.
Runs after the **Gate 2 GO** (`DECISIONS.md`, 2026-09-16). This is the E12–E13 batch; **E13 is
SKIPPED** (Gate 1 PIVOT), so the batch reduces to E12 alone. Execution stops at the batch boundary,
before **E14 (Gate 3)**.

**Objective.** Heteroskedastic (event-adaptive) intervals as a complement to split conformal —
testing whether adaptivity improves *efficiency* (narrower intervals at equal coverage).

**Method.** CQR on the **GBM quantile heads** (the E6 quantile capability; point-only learners have
no native quantiles and are out of E12 scope per the spec). Arms: naive CQR, rule-**weighted** CQR,
and CQR on the exchangeable **self-test** split (machinery validation, the E9 analog). Compared
per-event against the weighted split-conformal interval (E11-equivalent on the GBM point model) for
the width-efficiency analysis. Nominal {80,90,95}%, two-sided, 3 seeds, event-level bootstrap +
Clopper–Pearson CIs.

**Q-CONF-03 (non-blocking) resolved (b) for CQR:** the score is in-house (reusing the tested
finite-sample conformal quantile), not MAPIE/crepes — the weighted arm must combine it with the
in-house likelihood-ratio weights. Recorded in `DECISIONS.md`.

**Scope:** official test read once for scoring only; GBM point hyperparameters reused from the
Phase-2 cached search. Results reported exactly as observed (CLAUDE.md §3, §9); no gate call here.

In [ ]:
# --- Setup + provenance (invariant I4) ------------------------------------------------------
import json, subprocess, sys
from datetime import datetime, timezone

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal.models import conformal_runner as R

cfg = load_config()
FIGDIR = cfg.path("figures_dir"); FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha():
    try:
        return subprocess.run(["git","rev-parse","HEAD"], cwd=str(REPO_ROOT),
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return "UNAVAILABLE"

from kelvins_conformal.reporting import write_table_atomic
def save_table(df, name):
    write_table_atomic(df, TABDIR / f"{name}.csv"); print(f"saved: reports/tables/{name}.csv")
def save_fig(fig, name):
    for ext in ("png","pdf"): fig.savefig(FIGDIR/f"{name}.{ext}", dpi=160, bbox_inches="tight")
    print(f"saved: reports/figures/{name}.png|pdf")

PROVENANCE = {"experiment_ids": ["E12"], "git_commit_sha": git_sha(),
              "config_hash": cfg.config_hash, "seeds": list(cfg.train.seeds),
              "bootstrap_resamples": cfg.bootstrap.n_resamples,
              "executed_utc": datetime.now(timezone.utc).isoformat(), "python": sys.version.split()[0]}
print(json.dumps(PROVENANCE, indent=2))
CSPLIT, CCQR = "#0072B2", "#009E73"

## 1. Run E12

In [ ]:
RES = R.run_e12(cfg)
meta = RES["meta"]
print("seeds:", meta["seeds"], "| nominal:", meta["nominal_levels"],
      "| CQR quantile heads:", meta["cqr_quantile_levels"], "| n_supported:", meta["n_supported"])
cov = RES["coverage"]
for key in ("coverage", "widths"):
    save_table(RES[key], f"e12_{key}")
save_table(cov, "e12_coverage_all")

## 2. CQR machinery validation on the exchangeable self-split (the E9 analog)

Before trusting any official-test CQR number, validate the CQR machinery + GBM quantile heads where
exchangeability holds: on the self-test split, CQR should hit nominal. This distinguishes a
shift-driven official-test result from GBM quantile-head miscalibration (the cause E12's failure
criterion hypothesises).

In [ ]:
selfv = cov[cov.method=="E12_cqr_selftest"].sort_values("nominal")
display(selfv[["nominal","coverage_mean","coverage_sd","cp_lo_mean","cp_hi_mean","median_width_mean","n"]].round(4))
save_table(selfv, "e12_cqr_selftest")
# Exactness on exchangeable data: two-sided, because a deviation either way signals a defect
# (DECISIONS.md 2026-09-21, methodological note B1). Same form as before; now one named check.
self_checks = pd.concat([R.e12_coverage_checks(cov, lv) for lv in sorted(selfv["nominal"])])
self_checks = self_checks[self_checks["method"] == R.E12_SELFTEST_METHOD]
ok = bool(self_checks["passes"].all())
print(f"CQR self-test coverage tracks nominal at all levels "
      f"(exactness on exchangeable data - CP CI contains nominal): {ok}")
print("-> If True, the CQR machinery/quantile heads are sound; any official-test under-coverage")
print("   below is SHIFT-driven, not an E6 quantile-head bug.")

## 3. Coverage on the official (biased) test set — CQR (naive + weighted) vs split

E12 success criterion: CQR is coverage-valid AND narrower. Failure criterion: CQR fails valid
coverage even after weighting. Reported exactly as observed (CLAUDE.md §3, §9) — no tuning.

In [ ]:
def view(method):
    v = cov[cov.method==method].sort_values("nominal")
    return v[["nominal","coverage_mean","coverage_sd","gap_pp","cp_lo_mean","cp_hi_mean","median_width_mean","n"]].round(4)
for m in ("E12_cqr_naive","E12_cqr_weighted_rule","E11ref_split_weighted_rule"):
    print(f"=== {m} ==="); display(view(m))

prim = meta["primary_level"]
# Validity under shift is ONE-sided: coverage >= nominal, i.e. the CP CI upper bound is at or
# above nominal, so an over-covering arm is valid. (Corrected 2026-09-21: this used CI
# containment, the E17 H1 error class. It changed no label - see DECISIONS.md, note B4.)
chk = R.e12_coverage_checks(cov, prim).set_index("method")
print()
print(f"At nominal {prim:.0%}, validity under shift (one-sided: CP CI upper bound >= nominal):")
for m in R.E12_OFFICIAL_METHODS:
    print(f"  {m}: coverage {chk.loc[m, 'coverage']:.3f}  valid={bool(chk.loc[m, 'passes'])}")

## 4. Efficiency — interval width, CQR vs split

In [ ]:
w = RES["widths"]
display(w.round(3)); save_table(w, "e12_width_efficiency")
fig, ax = plt.subplots(figsize=(6.4,3.8))
x = np.arange(len(w)); bw=0.38
ax.bar(x-bw/2, w["split_median_width"], bw, label="split conformal (E11 ref)", color=CSPLIT)
ax.bar(x+bw/2, w["cqr_median_width"], bw, label="CQR (weighted)", color=CCQR)
ax.set_xticks(x); ax.set_xticklabels([f"{n:.0%}" for n in w["nominal"]])
ax.set_xlabel("nominal level"); ax.set_ylabel("median interval width (log10 risk)")
ax.set_title("E12 efficiency: CQR vs split-conformal median width"); ax.legend(frameon=False)
save_fig(fig, "e12_width_efficiency")
plt.show()
for _,r in w.iterrows(): print(f"  {r['nominal']:.0%}: CQR/split width ratio = {r['width_ratio_cqr_over_split']:.3f}")

## 5. Adaptivity — CQR width vs predicted risk (median seed, primary level)

In [ ]:
a = RES["adaptivity"]
fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(a["risk_last"], a["cqr_width"], s=8, alpha=0.4, color=CCQR, label="CQR width")
ax.axhline(a["split_width"].iloc[0], color=CSPLIT, ls="--", lw=1.4,
           label=f"split width (constant = {a['split_width'].iloc[0]:.1f})")
ax.set_xlabel("predicted risk level  r_last  [log10 Pc]"); ax.set_ylabel("interval width")
ax.set_title("E12 adaptivity: CQR interval width vs predicted risk"); ax.legend(frameon=False)
save_fig(fig, "e12_adaptivity")
plt.show()
print(f"CQR width std = {np.std(a['cqr_width']):.3f} (split width std = {np.std(a['split_width']):.3f})")
print(f"corr(CQR width, predicted risk) = {float(np.corrcoef(a['cqr_width'], a['risk_last'])[0,1]):.3f}")

## 6. E12 findings — measurement only, reported exactly as observed

In [ ]:
prim = meta["primary_level"]
# Official-test labels answer validity under shift (one-sided); the self-test label answers
# exactness on exchangeable data (two-sided). Both come from one named helper.
_chk = R.e12_coverage_checks(cov, prim).set_index("method")
def cov_at(method):
    return float(_chk.loc[method, "coverage"]), bool(_chk.loc[method, "passes"])
naive90, naive_valid = cov_at("E12_cqr_naive")
wtd90, wtd_valid = cov_at("E12_cqr_weighted_rule")
self90, self_valid = cov_at("E12_cqr_selftest")
split90, split_valid = cov_at("E11ref_split_weighted_rule")
wr = RES["widths"]["width_ratio_cqr_over_split"].mean()
print(f"""
E12 / CQR — WHAT THE BATCH SHOWS (measurement only, exactly as observed)

 EFFICIENCY (the clear win):
  * CQR intervals are ~{wr:.2f}x the width of split conformal at equal nominal (~{1/wr:.1f}x
    narrower), and ADAPTIVE — width tracks predicted risk (sec 5); split is constant-width.

 COVERAGE (a genuine negative — NOT smoothed over):
  * CQR machinery VALIDATED on the exchangeable self-split: {self90:.3f} at {prim:.0%}
    (CP CI contains nominal: {self_valid}). GBM quantile heads are sound and the CQR code path is
    correct -> E12's hypothesised failure cause (E6 quantile-head miscalibration) is REFUTED.
  * On the official (biased) test set CQR UNDER-COVERS: naive {naive90:.3f} (valid={naive_valid}),
    weighted {wtd90:.3f} (valid={wtd_valid}) at {prim:.0%}. This meets E12's failure criterion
    (invalid coverage even after weighting).
  * Weighting does NOT restore CQR coverage (weighted <= naive), whereas the SAME rule weights DID
    restore split conformal (E11 ref {split90:.3f}, valid={split_valid}). The selection-bias
    correction is effective for the absolute-residual (split) score but not the CQR score here.

 READING (for Sidh, not decided here): the official under-coverage is shift-driven (machinery
 validated), so the 'revisit' the failure criterion calls for should target the weight-vs-CQR-score
 interaction, NOT the E6 quantile heads (self-test exonerates them). Whether CQR enters the
 manuscript as an efficiency result with a stated coverage caveat, or is held pending a weighting
 fix, is Sidh's call.

 NOT DECIDED HERE (CLAUDE.md §3, §9, §13.7): that disposition; anything in E14/Phase 4.
 Execution stops at the E12-E13 batch boundary (E13 SKIPPED).
""")
(cfg.path("reports_dir")/"03b_cqr_provenance.json").write_text(json.dumps(PROVENANCE, indent=2), encoding="utf-8")
print("provenance:", cfg.path("reports_dir")/"03b_cqr_provenance.json")